## Title master thesis
# “Research Question”

---
**Authors:** Leonard Rampf, Niklas Keckeisen

**Advisors:** Daniel Obermeier, Batikas

**Submission Date:** 20.05.2026

# Table of Contents
- [1. Data Preparation](#3.-data-preparation)
  - [1.1 Data Loading](#3.1-data-loading)
  - [1.2 Data Cleaning](#3.2-data-cleaning)
  - [1.3. Investigation and Dropping of Irrelevant Columns](#3.3.-investigation-and-dropping-of-irrelevant-columns)
  - [1.4 Conversion of values](#3.4-conversion-of-values)

- [2. Exploratory data analysis (EDA)](#4.-exploratory-data-analysis-eda)
  - [2.1 Descriptive Statistics](#4.1-descriptive-statistics)
  - [2.2 Skewness and Kurtosis](#4.2-skewness-and-kurtosis)
  - [2.3 CDF's](#4.3-cdfs)
  - [4.4 Outliers](#4.4-outliers)
  - [4.5 Data Transformation](#4.5-data-transformation)
    - [4.5.1 Box-Cox Transformation](#4.5.1-box-cox-transformation)
    - [4.5.2 Log Transformation](#4.5.2-log-transformation)
    - [4.5.3 Verification of Transformation](#4.5.3-verification-of-transformation)
  - [4.6 Multivariate Overview & Interpretation](#4.6-multivariate-overview--interpretation)
  - [4.7 Preliminary Hypothesis Exploration](#4.7-preliminary-hypothesis-exploration)
    - [4.7.1 Income vs Job Level](#4.7.1-income-vs-job-level)
    - [4.7.2 Income vs Years Worked](#4.7.2-income-vs-years-worked)
    - [4.7.3 Income vs NumCompaniesWorked](#4.7.3-income-vs-numcompaniesworked)
    - [4.7.4 NumCompaniesWorked vs Job Satisfaction](#4.7.4-numcompaniesworked-vs-job-satisfaction)

- [5. Hypothesis 1](#5.-does-job-hopping-have-any-effect-on-income)

# Introduction

Short Introduction
Statement of Research Question

Hypothesis

# Data Preparation

## Import Statements

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

## Data Loading

In [4]:
relative_path = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_retrieval_results_geo_v2.csv")
input_file = os.path.abspath(relative_path)

df = pd.read_csv(input_file)

## Data Investigation

In [5]:
df.head()

,query_id,query,doc_id,method,doc_text,rank,is_target
0,50881_doc,holidaytraditions,50881_doc_comp_2,default_method,Name: HolidayTraditions State Trooper Personal...,1,0
1,50881_doc,holidaytraditions,50881_doc_comp_3,default_method,Name: Personalized Christmas Ornaments Family ...,2,0
2,50881_doc,holidaytraditions,50881_doc_comp_9,default_method,Name: Personalized Christmas Ornaments Family ...,3,0
3,50881_doc,holidaytraditions,50881_doc_TARGET,default_method,Name: Personalized Christmas Ornaments 2020 – ...,4,1
4,50881_doc,holidaytraditions,50881_doc_comp_1,default_method,Name: Kids Christmas Ornaments 2021 – Personal...,5,0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109875 entries, 0 to 109874
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   query_id   109875 non-null  object
 1   query      109875 non-null  object
 2   doc_id     109875 non-null  object
 3   method     109875 non-null  object
 4   doc_text   109875 non-null  object
 5   rank       109875 non-null  int64 
 6   is_target  109875 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 5.9+ MB


In [8]:
duplicate_rows = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

Number of duplicate rows: 0


In [9]:
print(f"Number of unique values in 'query_id': {df['query_id'].nunique()}")
print(f"Number of unique values in 'query': {df['query'].nunique()}")
print(f"Number of unique values in 'doc_id': {df['doc_id'].nunique()}")
print(f"Number of unique values in 'method': {df['method'].nunique()}")
print(f"Number of unique values in 'doc_text': {df['doc_text'].nunique()}")
print(f"Number of unique values in 'rank': {df['rank'].nunique()}")
print(f"Number of unique values in 'is_target': {df['is_target'].nunique()}")

Number of unique values in 'query_id': 11000
Number of unique values in 'query': 500
Number of unique values in 'doc_id': 109875
Number of unique values in 'method': 1
Number of unique values in 'doc_text': 15089
Number of unique values in 'rank': 10
Number of unique values in 'is_target': 2


## Data Preprocessing

First we will look at the 15 errors with missing target documents that were identified in the retrieval stage. Ghost targets are marked and penalized with a rank of k + 1, where k is the max number of documents in a query_id. 

In [16]:
# 1. FILE CONFIGURATION
jsonl_relative_path = os.path.join("..", "..", "data", "retail", "retrieval", "20260413_vertex_geoedits_v1.jsonl") 
jsonl_baseline_path = os.path.abspath(jsonl_relative_path)

def inject_ghost_targets_with_jsonl(df):
    print("Scanning for Missing Targets...")
    
    # 1. Find the missing queries
    all_queries = df['query_id'].unique()
    queries_with_targets = df[df['is_target'] == 1]['query_id'].unique()
    missing_target_queries = set(all_queries) - set(queries_with_targets)
    
    print(f"Found {len(missing_target_queries)} queries missing their Target document.")
    
    if len(missing_target_queries) > 0:
        # 2. Extract the true query text, doc_id, and doc_text from the JSONL
        print("Mapping original text and target details from JSONL baseline...")
        target_map = {}
        
        with open(jsonl_baseline_path, 'r', encoding='utf-8') as file:
            for line in file:
                record = json.loads(line.strip())
                
                doc_id = record.get("id", "")
                struct_data = record.get("structData", {})
                q_id = struct_data.get("query_id")
                
                if q_id in missing_target_queries:
                    if "TARGET" in doc_id.upper():
                        target_map[q_id] = {
                            "query": struct_data.get("original_query", "UNMAPPED"),
                            "doc_id": doc_id,
                            "text": struct_data.get("text", "N/A") 
                        }

        # 3. Create the penalized rows with true context
        ghost_rows = []
        for q_id in missing_target_queries:
            # Retrieve the mapped data, with a safety fallback
            mapped_data = target_map.get(q_id, {
                "query": "UNMAPPED", 
                "doc_id": "GHOST_TARGET_NOT_FOUND_IN_JSONL", 
                "text": "TEXT_NOT_FOUND"
            }) 
            
            ghost_rows.append({
                "query_id": q_id,
                "query": mapped_data["query"],
                "doc_id": mapped_data["doc_id"],       
                "method": "imputed_penalty",
                "doc_text": mapped_data["text"],       
                "rank": 11,                            # THE PENALTY
                "is_target": 1
            })
            
        ghost_df = pd.DataFrame(ghost_rows)
        
        # 4. Inject and Return
        df_final = pd.concat([df, ghost_df], ignore_index=True)
        print("Injection complete.")
        return df_final
        
    else:
        print("No missing targets found. The dataset is already perfectly intact.")
        return df
        
df2 = inject_ghost_targets_with_jsonl(df)

Scanning for Missing Targets...
Found 15 queries missing their Target document.
Mapping original text and target details from JSONL baseline...
Injection complete.


## Exploratory Data Analysis

In [10]:
df.describe().style.format('{:.2f}')

,rank,is_target
count,109875.00,109875.00
mean,5.49,0.10
std,2.87,0.30
min,1.00,0.00
25%,3.00,0.00
50%,5.00,0.00
75%,8.00,0.00
max,10.00,1.00
